# Fine-Tuning and Training Kite 1.0 in Google Colab

Welcome! This notebook provides a complete guide and execution pipeline to train/fine-tune the rebranded **Kite 1.0** Dynamic Multimodal MoE model on a standard **T4 GPU** (16GB VRAM) in Google Colab.

## Key Capabilities:
1. **Downscaled End-to-End Training**: Initialize and train a scaled-down version of the Kite model structure to test code execution on resource-limited hardware.
2. **Projector & Vision Fine-Tuning**: Keep the massive text backbone frozen and train only the Vision Tower and PatchMerger Multimodal Projector.

## Step 1: Clone the GitHub Repository
If you are running this notebook inside Google Colab, you need to clone your Kite repository and navigate into it to run the training scripts.

In [ ]:
# Replace the URL below with your actual GitHub repository URL
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPOSITORY.git"

import os
if not os.path.exists("train_kite.py"):
    print(f"Cloning {REPO_URL}...")
    !git clone {REPO_URL} kite_repo
    %cd kite_repo
    print("Switched working directory to:", os.getcwd())
else:
    print("Already inside the repository or files already present.")

## Step 2: Install Dependencies
Run the following cell to install the necessary packages for model instantiation, text generation, and video/image processing.

In [ ]:
# Install huggingface libraries, accelerate for model distribution, and decord/mecord for video handling
!pip install -q transformers accelerate peft decord pillow einops pydantic tiktoken

## Step 3: Set Up Environment & Mount Google Drive (Optional)
If you want to train on datasets stored in Google Drive or save checkpoints to Drive, run the cell below.

In [ ]:
# Mount Google Drive if you need to load data or save checkpoints permanently
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully!")
except ImportError:
    print("Running locally or outside Google Colab environment.")

## Step 4: Run Downscaled Verification Training
To verify that the dataset parsing, processor transformations, loss calculations, backpropagation, and weight updates work successfully, we can run `train_kite.py` in `--mode downscaled` with `--dummy` data.

This runs a downscaled config (1 layer, small embedding dimensions) that loads and trains on a single T4 GPU instantly.

In [ ]:
# Run training script locally using the downscaled config and dummy data
!python train_kite.py \
    --model_path "." \
    --mode "downscaled" \
    --dummy \
    --epochs 1 \
    --lr 2e-5 \
    --batch_size 1 \
    --grad_accum 2 \
    --output_dir "./kite_downscaled_checkpoint"

## Step 5: Fine-Tuning Projector / Vision Tower
To train only the projector and vision layers (freezing the language model), set `--mode projector_only`. 

*Note: Loading the full DeepSeek-V3 671B text backbone requires ~350GB of VRAM and will OOM on a T4. It is recommended to use a downscaled text config or configure a smaller base language model for training.*

In [ ]:
# Run training in projector-only mode
!python train_kite.py \
    --model_path "." \
    --mode "projector_only" \
    --dummy \
    --epochs 1 \
    --lr 2e-5 \
    --batch_size 1 \
    --grad_accum 4 \
    --output_dir "./kite_projector_checkpoint"

## Step 6: Preparing a Custom Dataset
To train on your own data, structure a JSON file containing a list of objects with image/video file paths and LLaVA-style conversations. 

Here is an example format (`dataset.json`):
```json
[
  {
    "image": "/path/to/image1.jpg",
    "conversations": [
      {"role": "user", "content": "<image>\nWhat does this image show?"},
      {"role": "assistant", "content": "It shows a custom model prediction."}
    ]
  }
]
```
Then run training by referencing it using `--data_path`:
```bash
!python train_kite.py --model_path "." --mode "downscaled" --data_path "/content/dataset.json" --epochs 5
```